### Question 1: Which markets are most efficient, and why?

The definition of market efficiency is how well the market bakes in new information into its prices - to what extent do the prices in the market reflect all available information, and how quickly does the market's prices adjust to new information?  A corollary of this is that it is very difficult for a trader to generate profits because the prices are already "fair value".  

In the case of assets such as stocks, efficiency means that an investor has a hard time beating the overall market benchmark's rate of appreciation (such as S&P500 index's long-term performance of about 7-9% annually).

In the case of daytrading (not investors, those who bet on short-term swings), market efficiency means that they cannot make consistent positive profits because their expected profit E(x) hovers around 0%, especially after transaction costs like broker fees and paying bid/ask spreads.

Generally, efficient markets are also highly liquid - they exhibit high trading volumes and narrow bid/ask spreads. This is because when a market has many participants trading in it, the higher chance that a subset of them with crucial information bakes that information into the market prices through their buying/selling actions, and also, the market responds faster to new information.  In other words, high liquidity is correlated with high efficiency, but they are not the same concept.  For instance, here is a degenerate edge case: a market might only have 2 participants, only one buyer and one seller - this is very illiquid.  But, let's say that the true value of a contract is &#36;0.7, and the buyer is bidding &#36;0.60, and the seller is offering at &#36;0.80, which comes out to a mid value of &#36;0.7.  This market is efficient, nonetheless, despite being illiquid.  But this is a very rare in the real financial markets, and I will still use liquidity as one measurement of efficiency.

For prediction markets: I would evaluate efficiency based on these criteria:

Criteria that we *can* test in this dataset:
- **High liquidity**, which has 3 sub bullet points: (1) The bid/ask spread is narrow, (2) the order book is deep, and (3) The trading volume is high. We can assess these criteria with the data that we are given. Which of these 33 markets has the narrowest bid/ask spread, and/or deepest order book, and/or the highest trading volume?
- **Is there arbitrage (risk-less profit opportunity) in the markets?** (4) More details below on the list of mathematical relationships that must hold, or else, an arbitrage exists.

Criteria that we *cannot* test in this dataset:
- **Does the price p of a contract actually resolve to "yes" in the long-term p percent of the time?** Given this small dataset, I cannot assert this.  If given time/resources, I would evaluate this by looking the history of baseball game prediction markets, checking the P(yes), and verify how many of those actually ended with resolving as yes (payoff = 1).
- **Is there a trading strategy that generates profits consistently?** With this small dataset, I cannot say this for sure. If given time/resources, I would try to build valuation models for the "fair value" of these contracts and verify this. I would also verify some simple models like momentum indicators or mean reversion, to test if there is auto-correlation in the time series of these prices.
- **Does new information get baked into the market quickly?** I don't have this information in this dataset.  But I would check this by looking at the history of news releases about baseball, i.e. - a star player on a team has been injured, or the betting market about team A vs. team B just received new information that team A beat a team C today, and see how fast these markets adjust accordingly to these news events.


### Let's read the data and evaluate these markets based on these criteria mentioned above

In [2]:
import numpy as np
import pandas as pd

from utils import (median_relative_trading_cost,   # (1) s / (p*(1-p)), median across the window
                   median_order_book_depth,        # (2) qty summed over both sides, median
                   total_contracts_traded)         # (3) qty summed over every trade

# Self-contained: this notebook re-reads the parquet files, so nothing else
# has to be run first.
df_books  = pd.read_parquet("data/orderbook_data_823753_pregame.parquet")
df_trades = pd.read_parquet("data/trades_823753_pregame.parquet")

# All 33 markets, with the shared game stamp stripped off for readability
MARKETS = sorted(df_books["native_id"].unique())
short   = lambda m: m.replace("-26AUG051940PITMIL", "")

print(f"books  {df_books.shape[0]:,} rows across {df_books['native_id'].nunique()} markets")
print(f"trades {df_trades.shape[0]:,} rows across {df_trades['native_id'].nunique()} markets"
      f"  ({len(MARKETS) - df_trades['native_id'].nunique()} markets never traded)")


books  58,275 rows across 33 markets
trades 2,232 rows across 31 markets  (2 markets never traded)


### (1) Trading costs of crossing bid/ask spread

Most of the bid/ask spreads in these markets are &#36;0.01, so for sake of argument, we'll go with that as our example. Assume that P(yes) = 0.9 in a market, which means P(no) = 0.1. If we buy yes at p = 0.9, we have a max loss of &#36;0.90. But selling no at &#36;0.1 is economically identical: we collect &#36;0.1 premium upfront, but we have a potential liability of &#36;1 if the bet goes badly, so max loss is also &#36;0.9 as well. Therefore:

**Buying yes** (equivalently, selling no) deploys $p$ of capital, so the relative cost is

$$\frac{s}{p} \;=\; \frac{0.01}{0.9} \;=\; 1.11\%$$

**Selling yes** (equivalently, buying no) deploys $1-p$ of capital, so the relative cost is

$$\frac{s}{1-p} \;=\; \frac{0.01}{0.1} \;=\; 10.0\%$$

If we want a **direction-neutral** measure of how expensive the market is to trade in bid/ask terms, we should consider both directions and sum them:

$$\frac{s}{p} \;+\; \frac{s}{1-p} \;=\; \frac{s(1-p) + sp}{p(1-p)} \;=\; \frac{s}{p\,(1-p)}$$

which for this example gives

$$\frac{0.01}{0.9 \times 0.1} \;=\; \frac{0.01}{0.09} \;=\; 11.11\% \;=\; 1.11\% + 10.0\% \quad \checkmark$$

Note that the two individual costs are not symmetric — the cheap side of the contract is the expensive side to trade — but their sum is, since $p(1-p)$ is unchanged when $p$ and $1-p$ swap places. That is what makes it a property of the *market* rather than of whichever direction we happened to pick.


Because trading costs can be generally summarized as the bid/ask + broker fees or commissions, the other component of trading costs is the commission/fee: p*(1-p)*0.07. Let's work that calculation out here:
- YES buyer deploys p, pays 0.07·p(1-p) → 0.07(1-p)
- NO buyer deploys (1-p), pays the same → 0.07p
- Sum = 0.07(1-p) + 0.07p = 0.07, exactly

 The fee is a flat/constant 7 percentage points on every one of the 33 markets, at every price level, if we add up the cost of "buying yes" and "selling yes" together. Not "higher near 0.5" (which happens if we measure from only one side - buy or sell).  Since that is an additive constant, including it cannot change the ranking, which is why I've omitted it from the calculation and noted it here instead.

Instruction to Claude for trading costs calculation:
- Use the books data (from orderbook_data_823753_pregame.parquet)
- Create a function for one market. In this function: per order book observation/photo (it doesn't matter if msg_type = update or snapshot, because we already established that both are a photo of the order book), calculate the trading cost using the formula above s/(p*(1-p)). For p, just use the mid, which is the average of best_bid and best_offer. For spread s, use difference between best_bid and best_ask: best_ask - best_bid. After you calculate this for every observation/photo of the order book, take the median of the trading cost
- Apply this function for all 33 markets

In [2]:
# (1) Median relative trading cost, one market at a time
cost = pd.Series(
    {short(m): median_relative_trading_cost(df_books[df_books["native_id"] == m])
     for m in MARKETS},
    name = "trading_cost",
)

# Report as a percentage of capital deployed; lower is better
(100 * cost).sort_values().round(2).to_frame("trading_cost_%")


,trading_cost_%
KXMLBTEAMTOTAL-MIL4,4.00
KXMLBF5TOTAL-4,4.00
KXMLBTEAMTOTAL-PIT4,4.05
KXMLBTOTAL-8,4.05
KXMLBTOTAL-7,4.09
KXMLBRFI,4.15
KXMLBTEAMTOTAL-MIL3,4.31
KXMLBTOTAL-9,4.31
KXMLBF5TOTAL-5,4.31
KXMLBTEAMTOTAL-MIL5,4.37


### (2) How deep is the order book?

This is a straightforward check: what is the sum of the quantities of the highest 5 bids and lowest 5 asks for each of these 33 markets?

Instruction to Claude for order book depth calculation:
- Use the books data (from orderbook_data_823753_pregame.parquet)
- Create a function for one market. In this function: per order book observation/photo (it doesn't matter if msg_type = update or snapshot, because we already established that both are a photo of the order book), calculate the sum of the quantities associated with the 5 bids and 5 asks. If the order book has less than 5 bids and 5 asks, just use whatever is available. An order book with only 2 or 3 levels of bids or asks is inherently less liquid, so we would capture that illiquid in the sum, which will be smaller as a result. After you calculate the sum of quantities of the order book for every observation/photo of the order book, take the median
- Apply this function for all 33 markets

In [3]:
# (2) Median order book depth, one market at a time
depth = pd.Series(
    {short(m): median_order_book_depth(df_books[df_books["native_id"] == m])
     for m in MARKETS},
    name = "depth",
)

# Higher is better
depth.sort_values(ascending = False).round(0).to_frame("median_depth")


,median_depth
KXMLBRFI,3232711.0
KXMLBTOTAL-8,444485.0
KXMLBTOTAL-7,340762.0
KXMLBTOTAL-9,283777.0
KXMLBTOTAL-6,190431.0
KXMLBF5TOTAL-4,132731.0
KXMLBTOTAL-10,114164.0
KXMLBTOTAL-11,107641.0
KXMLBTOTAL-12,102596.0
KXMLBTOTAL-5,97518.0


### (3) What is the trading volume in this market?

This is a straightforward check: what is the sum of the trading volume in this market during the timeframe of this dataset?

Instruction to Claude for trading volume calculation:
- Use the trade data (from trades_823753_pregame.parquet)
- Create a function for one market. Simply add up the quantities of all trades across all times in the trade dataframe, for this market.  Every time that you see a trade for this market, add the quantity to a running sum.  At the end, you will have a total number of contracts traded in this timeframe for this market
- Apply this function for all 33 markets.

In [4]:
# (3) Total quantity traded, one market at a time.
# Sliced from the TRADES frame, and a market with no trades at all yields 0.
volume = pd.Series(
    {short(m): total_contracts_traded(df_trades[df_trades["native_id"] == m])
     for m in MARKETS},
    name = "volume",
)

# Higher is better
volume.sort_values(ascending = False).round(2).to_frame("total_volume")


,total_volume
KXMLBRFI,272584.21
KXMLBTOTAL-8,98745.90
KXMLBF5TOTAL-4,35942.38
KXMLBTOTAL-7,28881.61
KXMLBTEAMTOTAL-MIL4,11829.93
KXMLBTOTAL-3,7041.81
KXMLBF5TOTAL-5,5928.34
KXMLBTOTAL-9,4045.00
KXMLBTOTAL-6,3289.39
KXMLBTOTAL-5,2386.03


### All three metrics side by side

One row per market. `trading_cost_%` is a cost, so lower is better; `median_depth`
and `total_volume` are both liquidity, so higher is better. Sorted by volume.

**Answer: KXMLBRFI is the most efficient market**, and it is not close. It is the deepest book by 7.3x over the runner-up (TOTAL-8) and 106x over the median market, and the highest volume by 2.8x over the runner-up and 519x over the median.

Trading cost does not decide this. RFI is 6th of 33 at 4.15%, against 4.00% for the cheapest — a 0.15 point gap that is really just tick granularity, since fifteen markets sit inside 4.00–4.50%. The spread is 1 cent nearly everywhere and p(1-p) is flat mid-ladder, so cost separates almost nothing. Depth and volume do all the ranking work.


In [5]:
pd.set_option("display.max_rows", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

q1 = pd.DataFrame({
    "trading_cost_%": 100 * cost,
    "median_depth":   depth,
    "total_volume":   volume,
})

q1.sort_values("total_volume", ascending = False)


,trading_cost_%,median_depth,total_volume
KXMLBRFI,4.15,"3,232,711.38","272,584.21"
KXMLBTOTAL-8,4.05,"444,484.61","98,745.90"
KXMLBF5TOTAL-4,4.00,"132,730.82","35,942.38"
KXMLBTOTAL-7,4.09,"340,761.74","28,881.61"
KXMLBTEAMTOTAL-MIL4,4.00,"28,801.27","11,829.93"
KXMLBTOTAL-3,23.27,"91,512.52","7,041.81"
KXMLBF5TOTAL-5,4.31,"44,138.63","5,928.34"
KXMLBTOTAL-9,4.31,"283,777.11","4,045.00"
KXMLBTOTAL-6,4.43,"190,431.31","3,289.39"
KXMLBTOTAL-5,5.73,"97,517.51","2,386.03"


### (4) Checking for Arbitrage

- **Markets have decreasing probabilities p, as number of runs n goes up**: Getting at least 5 runs in a game cannot be more likely than getting at least 4 runs in a game.  Generally speaking, P(runs >= n) is a survival function and must be non-increasing (and most likely decreasing) in number of runs n.  Proof by contradiction: if S(5) = &#36;0.80 and S(4) = &#36;0.75, then we can buy the strike 4 at &#36;0.75, sell the strike 5 at &#36;0.80, earn &#36;0.05 upfront on day 1 (no matter how many runs happen), and if there are exactly 4 runs, we also earn an extra &#36;1 because the 4-strike pays out and the 5-strike does not.  Thus, as a test of market efficiency, we need to check that all of the markets have decreasing probabilities as number of runs n goes up

- **P(KXMLBRFI) ≤ P(KXMLBF5TOTAL runs = 1)**.  
KXMLBRFI = Will there be at least 1 run in the 1st inning of a baseball game (both teams combined total).  
KXMLBF5TOTAL - # of runs in the first 5 innings of the game (both teams combined total).  
Thus, at strike runs = 1 for KXMLBF5TOTAL, we are measuring the probability of there being at least 1 run in the first 5 innings of the game.  This must be at least as great as the probability of at least 1 run in the first inning of the game. 
- **P(KXMLBF5TOTAL) ≤ P(KXMLBTOTAL) for all strikes n**
KXMLBF5TOTAL - # of runs in the first 5 innings of the game (both teams combined total)
KXMLBTOTAL - # of runs scored in the game (both teams combined total)
Same logic as before
- **KXMLBTOTAL >= max[KXMLBTEAMTOTAL(team A), KXMLBTEAMTOTAL(team B)]** In other words, the probability that sum of both teams' runs add up to at least 5 must be at least as great as team A's chance of getting at least 5 runs, and at least as great as team B's chance of getting at least 5 runs

### The background / reasoning for the code & results that follow

First, let's make a list of chains that we need to check for monotonicity:
- KXMLBF5TOTAL (all strikes against each other)
- KXMLBTEAMTOTAL (all strikes of team MIL against each other)
- KXMLBTEAMTOTAL (all strikes of team PIT against each other)
- KXMLBTOTAL (all strikes against each other)

A total of 4 ladders/chains to compare.  Compare probabilities for different pairs of strikes in each chain, repeat across 4 chains
- In our code, we need to define 4 chains of strikes, such that when we do comparisons, we know which securities are in the same chain/family

The problem is: order book snapshots of adjacent strikes don't appear at the same time. So, the question is: given the quotes are stale often, even for adjacent strikes in the same chain, what do we do?

For example, the time between F5TOTAL-5 order book snapshot to the next F5TOTAL-6 order book snapshot has a distribution with:
- within 0.1s: 36.3%
- p50:    1.26 s
- p75:    7.40 s
- p90:   23.78 s
- p99:   72.11 s
- max:  119.25 s

One idea that I have is: define a threshold of time that makes a prior order book quote "too stale".  We can set it at 1 second for now. So what we can do is:
- Sort the order book timestamps in order, from earliest (oldest) to latest (most recent)
- Iterate through the order book dataframe, in chronological order, from earliest timestamp to latest timestamp. Use received timestamp for now because that's when we actually know the data on our side
- Every time that we see an order book for a security/market, we store it in a data structure (dict or nested list or whatever), to hold that information.  So we have a collection of "recent snapshots of order book securities"
- Key the cache by market, holding only the latest quote. If F5TOTAL-5 updates three times inside the window you want the newest, not three entries. A dict keyed on native_id does this naturally; a list would double-count.
- We move forward in time, to find the next order book. If that time is more than a threshold (1 second in this example) later than any of those recent snapshots, those snapshots get thrown out / removed.  Only the recent snapshots that are within 1 second of the current order book AND they can be compared, we will do the montonic comparison, to test for market efficiency

Example:
- In our recent order books, we have F5TOTAL-5 order book at 11:00:00 AM.
- Next order book is F5TOTAL-7, at 11:00:01 AM.  That is within 1 second of each other, so F5TOTAL-5 is still considered "fresh". They are in the same family / same chain, so we can compare is P(F5TOTAL-7) < P(F5TOTAL-5)?

Example:
- In our recent order books, we have F5TOTAL-5 order book at 11:00:00 AM.
- Next order book is F5TOTAL-8, at 11:00:05 AM.  5 seconds have elapsed - the F5TOTAL-5 order book is too stale.  Comparison is meaningless.  Skip

Example:
- In our recent order books, we have KXMLBTEAMTOTAL-PIT5 order book at 11:00:00 AM.
- Next order book is KXMLBTEAMTOTAL-MIL6, at 11:00:01 AM.  1 second has elapsed: KXMLBTEAMTOTAL-PIT5 is still fresh.  But one is from the PIT team chain and the other is from MIL team chain - different chains, so comparison is useless.  Skip

For the comparison on price, for now, use mid = (best_bid + best_ask)/2.  If we cannot find violations for mid prices, there is no way in hell that we will find violations after accounting for "buying on the bid / selling on the offer" + paying p*(1-p)*0.07 in execution fees.  If we find theoretical violations based on mid of strike A vs. mid of strike B in the same chain, then we have a shot at actually quantifying if this arbitrage is actually executable after costs

We can step through time, and for any order book, basically check if any recent (fresh) order book snapshots from the same chain can be compared to it, for monotonicity.  Report the results if any cases violate monotonicity (arbitrage exists).

We can also customize this function to run for threshold = 2 seconds or 5 seconds, etc. To see how it works for different definitions of freshness

Note on why mid prices are a valid screen and not just a shortcut: an executable
arbitrage needs bid(m) > ask(n) for n < m.  Since ask(n) > mid(n) and mid(m) > bid(m),
that chain gives mid(m) > bid(m) > ask(n) > mid(n).  So every executable violation is
also a mid violation -- mid violations are a strict superset.  If we find none on mids,
we have proven there are none that are executable, and we can stop there.

### Generalizing this to all four arbitrage conditions

Rather than hardcoding "are these two markets in the same chain?", build a lookup of
ordered pairs, each carrying the direction the constraint requires.  Built once, up
front:

1. Within-chain monotonicity, across all 4 chains: for every pair of strikes n < m in
   the same chain, require S(m) <= S(n)
2. RFI nests inside F5, testable at the one strike they share, n = 1:
   require S_RFI(1) <= S_F5(1)
3. F5 nests inside the whole game, at the strikes they share, n = 2 to 7:
   require S_F5(n) <= S_TOTAL(n)
4. Either team's runs are part of the game total, at shared strikes n = 2 to 8:
   require S_MIL(n) <= S_TOTAL(n), and S_PIT(n) <= S_TOTAL(n)

The engine is then one loop with no special cases: on each arriving order book, for
every fresh quote still in the cache, look up whether that pair has a required
direction, and if it does, test it.  The PIT-vs-MIL example above skips automatically,
because that pair simply has no entry in the lookup.

### What we record for each violation

Fields to capture per violation:
- timestamp of the earlier quote, and of the later quote
- the gap between them in seconds
- the two market names, and the direction the constraint required
- both mid prices
- the size of the violation, in cents
- the freshness threshold in force when it was caught

### Descriptive log

On top of the table, emit a human-readable line per violation so anomalies can be read
rather than only counted.  Something in the shape of:

    11:03:42.115  S_RFI(1)      mid 0.412
    11:03:43.002  S_F5TOTAL(1)  mid 0.405   (0.887 s later)
    -> violates S_RFI(1) <= S_F5(1) by 0.7c

If the tally comes back clean, that log is empty and the finding is simply "no
violations found at any threshold", which is the answer we should expect from a
reasonably efficient set of markets.

### (4) Arbitrage check -- results

Implemented per the spec above. `check_arbitrage_violations` walks the book feed once
in `recv_ts_utc` order, holds a cache of the latest quote per market, evicts anything
older than the freshness threshold, and tests every pair with a known constraint
direction. All four condition families are covered by one lookup of ordered pairs.


In [6]:
from utils import (check_arbitrage_violations,
                   arbitrage_sensitivity,
                   build_constraint_pairs,
                   market_short_name)

# The 139 constraints being enforced, grouped by family
pairs = build_constraint_pairs(sorted(df_books["native_id"].map(market_short_name).unique()))
fam = pd.Series([v.split(":")[0].split(" at ")[0] for v in pairs.values()]).value_counts()
print(f"{len(pairs)} ordered constraint pairs:\n")
print(fam.to_string())


139 ordered constraint pairs:

TOTAL monotonicity                   55
F5TOTAL monotonicity                 21
TEAMTOTAL-MIL monotonicity           21
TEAMTOTAL-PIT monotonicity           21
MIL runs nest in the game total       7
PIT runs nest in the game total       7
innings 1-5 nest in the full game     6
inning 1 nests in innings 1-5         1


**Sanity checks first.** A detector that always returns zero is indistinguishable from
a broken one, so before trusting a null result we corrupt the data on purpose and
confirm the check fires.

In [7]:
# POSITIVE CONTROL -- push S(8) above S(7) and confirm it is caught
bad = df_books.copy()
bad["short"] = bad["native_id"].map(market_short_name)
m8 = bad["short"] == "KXMLBTOTAL-8"
bad.loc[m8, "best_bid"], bad.loc[m8, "best_ask"] = 0.90, 0.91

ctrl = check_arbitrage_violations(bad, freshness_seconds = 1.0)
print(f"corrupted TOTAL-8  ->  {ctrl['n_violations']:,} violations detected\n")
print(ctrl["log"][0])

# NEGATIVE CONTROL -- MIL and PIT are different random variables, so no
# constraint may ever link them
cross = [(a, b) for a, b in pairs
         if "TEAMTOTAL" in a and "TEAMTOTAL" in b and ("MIL" in a) != ("MIL" in b)]
print(f"\nMIL-vs-PIT constraint pairs: {len(cross)}  (must be 0)")


corrupted TOTAL-8  ->  13,160 violations detected

17:35:35.010  S_TOTAL(8)   mid 0.905
17:35:35.011  S_TOTAL(7)   mid 0.575   (0.000 s later)
-> violates TOTAL monotonicity: S(8) <= S(7) by 33.0c

MIL-vs-PIT constraint pairs: 0  (must be 0)


**Now the real data.**

In [8]:
result = check_arbitrage_violations(df_books, freshness_seconds = 1.0)

print(f"constraints enforced : {result['n_constraints']}")
print(f"comparisons made     : {result['n_comparisons']:,}")
print(f"violations found     : {result['n_violations']:,}")
print()
print("\n".join(result["log"]) if result["n_violations"] else "(log is empty -- no anomalies to report)")


constraints enforced : 139
comparisons made     : 170,468
violations found     : 0

(log is empty -- no anomalies to report)


**Sensitivity to the freshness threshold.** If violations only showed up at loose
thresholds they would be artifacts of the feed going quiet rather than real market
failures. Here nothing appears at any threshold, including one so loose (300s) that it
is not a defensible measure of simultaneity at all.

In [9]:
arbitrage_sensitivity(df_books, thresholds = (1.0, 2.0, 5.0, 30.0, 300.0)).round(3)


,comparisons,violations,violation_%,worst_cents,tightest_slack_cents
threshold_s,,,,,
1.00,170468,0,0.00,0.00,1.00
2.00,206793,0,0.00,0.00,1.00
5.00,275714,0,0.00,0.00,1.00
30.00,447168,0,0.00,0.00,1.00
300.00,570567,0,0.00,0.00,1.00


**How close did it ever get?** "No violations" is only meaningful if the strikes were
plausibly near each other. This is the tightest the ladder ever came to breaking each
rule, in cents of remaining room.

In [10]:
result["slack"].head(15).round(2)


,tightest_slack_cents
rule,
TOTAL monotonicity: S(3) <= S(2),1.00
TEAMTOTAL-PIT monotonicity: S(8) <= S(7),4.50
TEAMTOTAL-MIL monotonicity: S(8) <= S(7),4.50
TOTAL monotonicity: S(11) <= S(10),5.00
TEAMTOTAL-MIL monotonicity: S(7) <= S(6),6.00
TOTAL monotonicity: S(12) <= S(11),6.00
TOTAL monotonicity: S(5) <= S(4),7.00
TEAMTOTAL-PIT monotonicity: S(7) <= S(6),7.00
F5TOTAL monotonicity: S(7) <= S(6),7.00


#### What this shows

**No arbitrage violations, anywhere.** 170,468 comparisons across 139 constraints at a
1-second freshness threshold, and 570,567 comparisons at 300 seconds. Zero violations at
every threshold, on every rule, in all four condition families.

Because mid violations are a strict superset of executable ones -- `bid(m) > ask(n)`
implies `mid(m) > mid(n)` -- this **proves** there was no executable arbitrage, without
needing to model the spread or the `p(1-p)*0.07` fee at all.

The slack table says the ladders were not merely ordered, they were ordered with room
to spare: the tightest any rule ever came was 1.0c, on `S(3) <= S(2)` in the game total,
where both contracts sit near 0.95 and there is little probability left between them.
Everything else kept 4.5c or more.

**The thin markets hold up too, and that is the surprise.** TEAMTOTAL-PIT and
TEAMTOTAL-MIL are the least liquid markets in the dataset -- widest costs, thinnest
books, and two of them never traded at all -- yet their ladders are internally
consistent and correctly nested inside the game total. Whoever is quoting them is
pricing a coherent distribution rather than posting numbers market by market. That is
evidence of a market maker running a model across the whole chain, and it separates
"illiquid" from "inefficient" in exactly the way the definition at the top of this
notebook requires.

**Caveat, and it is a real one.** 66% of trades land inside a gap of more than 5 seconds
in the book feed, so the feed demonstrably does not report every change. A violation
that opened and closed inside one of those gaps would be invisible to us. The freshness
threshold limits how badly this can mislead -- at 1 second we are only ever comparing
quotes we actually saw -- but it cannot rule out arbitrage in the dark.


### What this does not establish. 

This is one game, 33 markets, a 5.99-hour pregame window, one venue. It shows these markets were free of mid-to-mid arbitrage over this slice; it says nothing about the base rate across thirty days and hundreds of markets, where occasional violations would not be surprising — most likely visible on mids but not executable after crossing the spread and paying the fee.

The mechanism behind the clean result is also not identified here. The tightest pair, S(2) − S(3), sat at its 1¢ floor for 8.8% of the window and then widened with no trades in either leg in the prior 60 seconds — consistent with market makers skewing quotes to defend the gap rather than taking liquidity aggressively. But that is a single episode, so it cannot be distinguished from one maker refitting a distribution across the chain.

### Sidenote (considered, not computed): speed of repricing

I classified "does new information get baked in quickly?" as untestable here because
there is no news feed in this dataset. That is true for external news, but it is not
the whole story, and I want to flag what I would do with more time.

We have 58,275 book updates across 33 markets that are mechanically related to each
other -- the TOTAL strikes form one ladder, and F5 nests inside TOTAL. So a large
trade landing on one strike is itself an information event, and I can watch whether
the neighbouring strikes reprice, and how many seconds it takes.

That would be the only measure here that tests **efficiency** rather than
**liquidity**. Spread, depth and volume all describe how easy a market is to trade;
none of them show that prices actually absorb information. Repricing speed does, and
it is much harder to fake -- you can post tight quotes in a market nobody watches, but
you cannot make unrelated strikes move in sympathy unless somebody is genuinely
arbitraging them.

Not computed, for time reasons. Noting it so it is clear the omission is a scoping
decision rather than an oversight.
